<a href="https://colab.research.google.com/github/YashRYadav/Assignment/blob/main/English_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, normalized_mutual_info_score, adjusted_rand_score

print("Ensuring raw data is updated...")

# Simulated dataset of digital forensic incident reports
raw_data = [
    "Phishing\tMultiple users reported receiving fraudulent emails claiming to be from the IT helpdesk.",
    "Phishing\tA targeted spear-phishing campaign hit the finance department with malicious PDF attachments.",
    "Malware\tRansomware encrypted the main database server, demanding cryptocurrency for the decryption key.",
    "Malware\tThe endpoint detection system caught a trojan attempting to exfiltrate system configuration files.",
    "Intrusion\tUnauthorized SSH access was detected originating from an unknown offshore IP address.",
    "Intrusion\tThe firewall logs show repeated brute-force login attempts on the administrative web portal."
]

true_labels_text = []
corpus = []

# Parse the raw data into ground-truth labels and the text corpus
for line in raw_data:
    label, text = line.strip().split('\t')
    true_labels_text.append(label)
    corpus.append(text)

# Convert categorical string labels into numerical indices for evaluation metrics
unique_labels = list(set(true_labels_text))
true_labels = [unique_labels.index(label) for label in true_labels_text]
num_clusters = len(unique_labels)

print(f"Final raw data quantity: {len(raw_data)}")
print(f"Final text corpus quantity: {len(corpus)}")
print(f"Final true label quantity: {len(true_labels)}")
print(f"Final unique category quantity: {num_clusters}\n")

# Step 1: Feature Extraction (TF-IDF)
print("Extracting TF-IDF features...")
vectorizer = TfidfVectorizer(max_df=0.9, min_df=1, max_features=5000, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(corpus)

# Step 2: Dimensionality Reduction (Truncated SVD)
print("Performing SVD dimensionality reduction (Targeting 75% information retention)...")
max_components = min(tfidf_matrix.shape[0] - 1, tfidf_matrix.shape[1] - 1)

if max_components > 0:
    temp_svd = TruncatedSVD(n_components=max_components, random_state=42)
    temp_svd.fit(tfidf_matrix)
    cumulative_variance = np.cumsum(temp_svd.explained_variance_ratio_)
    target_variance = 0.75
    actual_n_components = np.argmax(cumulative_variance >= target_variance) + 1
    print(f"To retain {target_variance*100}% of information, optimal n_components = {actual_n_components}")
    svd = TruncatedSVD(n_components=actual_n_components, random_state=42)
    reduced_matrix = svd.fit_transform(tfidf_matrix)
else:
    print("Warning: TF-IDF matrix does not have enough features or samples. Cannot perform SVD.")
    reduced_matrix = tfidf_matrix.toarray()

# Step 3: K-Means Clustering
print("\nExecuting K-Means clustering...")
if reduced_matrix.shape[0] > 0 and reduced_matrix.shape[1] > 0:
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    predicted_labels = kmeans.fit_predict(reduced_matrix)
else:
    print("Warning: No data to perform K-Means clustering.")
    predicted_labels = np.array([])

# Step 4: Academic Evaluation Metrics
print("\n--- Clustering Results Evaluation ---")
if predicted_labels.size > 0 and reduced_matrix.shape[0] > 1:
    silhouette = silhouette_score(reduced_matrix, predicted_labels)
    print(f"Silhouette Score: {silhouette:.4f}")

    # NMI and ARI measure the agreement between the algorithmic clusters and the ground truth (Supervised)
    nmi = normalized_mutual_info_score(true_labels, predicted_labels)
    ari = adjusted_rand_score(true_labels, predicted_labels)
    print(f"Normalized Mutual Information (NMI): {nmi:.4f}")
    print(f"Adjusted Rand Index (ARI): {ari:.4f}")
else:
    print("Cannot calculate evaluation metrics, insufficient data or incorrect dimensions.")

# Step 5: Output Validation
print("\n--- Sample Comparison ---")
if predicted_labels.size > 0:
    for i in range(len(corpus)):
        print(f"True Category: {true_labels_text[i]} | Algorithm Assigned Cluster ID: {predicted_labels[i]}")
else:
    print("Cannot perform sample comparison, no clustering results.")